# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the [FAIR^2 Mass Spectrometry vesicle dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema](https://mlcommons.org/croissant/) URL.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant metadata schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print main metadata information
md = dataset.metadata
print(f"Dataset Name: {md.name}\n")
print(f"Description: {md.description}\n")
print(f"Published: {md.datePublished}")
print(f"Version: {md.version}")
print(f"License: {md.license}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id` values as defined in the Croissant metadata.

**Note**: The `@id` uniquely identifies each entity within the dataset: record sets, fields and columns.

In [ ]:
# List all available record sets and their fields using their @id
from pprint import pprint

# List all record set @id's
print("Record Sets in Dataset:")
record_sets = list(dataset.record_sets.keys())
for rs_id in record_sets:
    record_set = dataset.record_sets[rs_id]
    print(f"- RecordSet @id: {rs_id}")
    print(f"  Name: {record_set.name}")
    # List fields for this RecordSet
    print("  Fields (@id and name):")
    for field_id, field in record_set.fields.items():
        print(f"    - {field_id}: {field.name}")
    print("")

Let's inspect a few records from the main data table. For this dataset, the main record set is typically the proteomics table.

> **TIP**: Replace `<record_set_id>` with the `@id` you choose to explore (see previous cell output).

In [ ]:
# Example: print a few records from the first record set
if len(record_sets) == 0:
    print("No record sets found in this dataset.")
else:
    rs_id = record_sets[0]  # Take the first RecordSet
    print(f"Sample records from RecordSet: {rs_id}")
    for i, rec in enumerate(dataset.records(record_set=rs_id)):
        pprint(rec)
        if i >= 2:
            break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

- Use the RecordSet `@id` you want to extract.
- All columns/fields are referenced by their `@id`.
- We'll construct a DataFrame for each RecordSet.

In [ ]:
# Load all record sets as DataFrames, keyed by their @id
dataframes = {}

for rs_id in record_sets:
    print(f"\nLoading records from RecordSet @id: {rs_id}")
    recs = list(dataset.records(record_set=rs_id))
    if len(recs) > 0:
        df = pd.DataFrame(recs)
        dataframes[rs_id] = df
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(3))
    else:
        print("(No records in this RecordSet)")

## 4. Exploratory Data Analysis (EDA)
Apply basic processing: 
- Select a numeric field by its `@id` (e.g. abundance, peptide count, MW).
- Filter, normalize, and group by a categorical field.

In [ ]:
# Example selection -- update these with the @id values you want from previous outputs:

# Let's use the first record set for analysis
main_rs_id = record_sets[0] if record_sets else None
if main_rs_id is not None and main_rs_id in dataframes:
    df = dataframes[main_rs_id]
    # Try to guess some numeric and group fields
    
    numeric_candidates = [col for col in df.columns if df[col].dtype.kind in 'fi' or df[col].astype(str).str.fullmatch(r'\d+(\.\d+)?').sum() > 0]
    print(f"Potential numeric fields: {numeric_candidates}")
    
    # Let's select the first numeric candidate (update as needed)
    numeric_field = numeric_candidates[0] if numeric_candidates else None
    if numeric_field is not None:
        # Ensure numeric dtype
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Records with {numeric_field} > mean ({threshold:.2f}): {len(filtered_df)}")
        # Normalize field
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Try grouping by a likely field
        group_candidates = [col for col in df.columns if df[col].dtype.name == 'object' and col != numeric_field]
        if group_candidates:
            group_field = group_candidates[0]
            grouped = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Mean {numeric_field} grouped by {group_field}:")
            print(grouped.head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric field found in main RecordSet.")
else:
    print("No main dataframe to analyze.")

## 5. Visualization
Visualize distributions or relationships between fields in the dataset.

Plots may include:
- Histograms of a numeric field
- Boxplots or scatter plots
- Grouped bar plots by category

You may customize the visualization as needed by referencing fields using their `@id`.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if main_rs_id is not None and main_rs_id in dataframes and numeric_field is not None:
    df = dataframes[main_rs_id]
    plt.figure(figsize=(8,5))
    df[numeric_field].hist(bins=30, color='cornflowerblue', edgecolor='k')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.title(f'Distribution of {numeric_field}')
    plt.show()
    # If a group_field is present, a boxplot
    if 'group_field' in locals():
        if group_field in df.columns:
            plt.figure(figsize=(10,6))
            df.boxplot(column=numeric_field, by=group_field, grid=False, rot=45)
            plt.title(f"{numeric_field} by {group_field}")
            plt.suptitle("")
            plt.xlabel(group_field)
            plt.ylabel(numeric_field)
            plt.tight_layout()
            plt.show()

## 6. Conclusion

In this notebook, you explored the FAIR\u2072 proteomics dataset schema with [`mlcroissant`](https://github.com/mlcommons/croissant), loaded records into pandas, and performed basic data analysis.

- The tables, fields, and records were referenced by their unique `@id` as defined in the Croissant specification.
- You can filter, group, and visualize any field by referencing its `@id` from the overview section.

**Next steps**: Apply advanced analyses, machine learning, or further processing tailored to your scientific or biomedical research question.